# Clase 181 — Comparaciones múltiples: Bonferroni, Holm, FDR

Hacer muchos tests infla los falsos positivos. Cuantificamos la inflación de α y aplicamos las dos familias de corrección: **FWER** (Bonferroni, Holm) y **FDR** (Benjamini-Hochberg).

Requiere: `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`.

## 1. Inflación de α

Con `m` tests independientes y `H₀` verdadera, `P(≥1 falso positivo) = 1 - (1-α)^m`. Lo verificamos por simulación.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

m = 20
n_exp = 2000
any_sig = 0
for _ in range(n_exp):
    pvals = [stats.ttest_ind(rng.normal(0, 1, 30), rng.normal(0, 1, 30)).pvalue for _ in range(m)]
    any_sig += (min(pvals) < 0.05)
emp = any_sig / n_exp
theo = 1 - 0.95 ** m
print(f"P(≥1 falso positivo): empírico={emp:.3f}  teórico={theo:.3f}")
assert abs(emp - theo) < 0.05

## 2. Bonferroni

`α_local = α/m`, o equivalentemente `p_adj = min(p·m, 1)`. Controla FWER exactamente pero es conservador.

In [ ]:
pvals = np.array([0.001, 0.008, 0.02, 0.03, 0.04, 0.045, 0.05, 0.06, 0.1, 0.2,
                  0.3, 0.35, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 0.95])
bonf_manual = np.minimum(pvals * len(pvals), 1)
rej_b, adj_b, _, _ = multipletests(pvals, alpha=0.05, method="bonferroni")
print(f"Bonferroni rechaza {rej_b.sum()} de {len(pvals)}")
assert np.allclose(bonf_manual, adj_b)

## 3. Holm

Step-down: uniformemente más poderoso que Bonferroni sin perder control del FWER.

In [ ]:
rej_h, adj_h, _, _ = multipletests(pvals, alpha=0.05, method="holm")
print(f"Bonferroni: {rej_b.sum()} rechazos   Holm: {rej_h.sum()} rechazos")
assert rej_h.sum() >= rej_b.sum()

## 4. Benjamini-Hochberg (FDR)

Controla la **proporción esperada de falsos positivos entre los rechazos**. Simulamos genómica: 950 nulos + 50 alternativos.

In [ ]:
m0, m1 = 950, 50
null_p = rng.uniform(0, 1, m0)
alt_p = stats.beta.rvs(0.5, 8, size=m1, random_state=rng)   # concentrados cerca de 0
allp = np.concatenate([null_p, alt_p])
is_alt = np.concatenate([np.zeros(m0, bool), np.ones(m1, bool)])

rej_bh, _, _, _ = multipletests(allp, alpha=0.05, method="fdr_bh")
descubiertos = rej_bh.sum()
fdr_emp = rej_bh[~is_alt].sum() / max(descubiertos, 1)
recall = rej_bh[is_alt].sum() / m1
print(f"BH: {descubiertos} rechazos  recall={recall:.2f}  FDR empírico={fdr_emp:.3f}")
assert fdr_emp < 0.15

## 5. Comparación: poder vs control de errores

BH descubre mucho más que Holm/Bonferroni manteniendo el FDR acotado.

In [ ]:
rows = []
for meth in ("bonferroni", "holm", "fdr_bh"):
    rej, _, _, _ = multipletests(allp, alpha=0.05, method=meth)
    fdr = rej[~is_alt].sum() / max(rej.sum(), 1)
    rec = rej[is_alt].sum() / m1
    rows.append((meth, int(rej.sum()), round(rec, 3), round(fdr, 3)))
tab = pd.DataFrame(rows, columns=["método", "#rechazos", "recall", "FDR_emp"]).set_index("método")
print(tab)
assert tab.loc["fdr_bh", "#rechazos"] >= tab.loc["holm", "#rechazos"] >= tab.loc["bonferroni", "#rechazos"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(tab.index, tab["recall"], color=["#c33", "#e83", "#3a7"])
ax.set_ylabel("recall (poder)"); ax.set_title("BH descubre más manteniendo FDR≈5%")
plt.tight_layout(); plt.show()

## Ejercicios

1. Aplicá también Šidák (`method='sidak'`) al vector `pvals` y compará con Bonferroni.
2. Repetí el experimento del bloque 4 con `m1=200` alternativos y observá cómo cambia el recall de cada método.
3. Con features correlacionadas, usá `method='fdr_by'` (Benjamini-Yekutieli) y compará cuántos rechaza frente a `fdr_bh`.

## Conclusiones

- Sin corrección, `m` tests inflan los falsos positivos: `1 - 0.95^20 ≈ 64 %`.
- **FWER** (Bonferroni/Holm) cuando un solo falso positivo es costoso (medicina, seguridad); Holm domina a Bonferroni.
- **FDR** (BH) para screening masivo: tolera algunos falsos positivos para no perder descubrimientos verdaderos.
- Pre-especificá la corrección: elegirla después de ver los resultados también es p-hacking.

## ✅ Soluciones de los ejercicios
<!--SOL175184-->

Soluciones trabajadas y **ejecutables** de todos los ejercicios de la sección *🧪 Ejercicios*. Datos sintéticos reproducibles con `np.random.default_rng(42)`; sin dependencias de internet. Cada bloque incluye `assert`/`print` para autocorregir.

<!--SOL175184-->

**Ej. 1 — Inflación de α.** 5 000 experimentos × 20 tests con `H0` verdadera; ≈ 64 %.

In [ ]:
# <!--SOL175184-->
import numpy as np
from scipy import stats
rng = np.random.default_rng(42)
n_exp, m, n = 5_000, 20, 30
A = rng.normal(0, 1, (n_exp, m, n)); B = rng.normal(0, 1, (n_exp, m, n))
pmat = stats.ttest_ind(A, B, axis=2).pvalue
frac = (pmat < 0.05).any(axis=1).mean(); teo = 1 - 0.95**m
print(f"% con >=1 falso positivo: {frac:.1%} (teorico {teo:.1%})")
assert abs(frac - teo) < 0.03
print("Inflacion de alpha ~64%: OK")

<!--SOL175184-->

**Ej. 2 — Bonferroni** manual vs `multipletests`.

In [ ]:
# <!--SOL175184-->
from statsmodels.stats.multitest import multipletests
pvals = np.array([0.001,0.008,0.02,0.03,0.04,0.045,0.05,0.06,0.07,0.09,0.11,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95])
manual = np.minimum(pvals*len(pvals), 1.0)
rej_b, adj_b, _, _ = multipletests(pvals, alpha=0.05, method="bonferroni")
assert np.allclose(manual, adj_b)
print(f"Bonferroni manual == statsmodels (rechaza {rej_b.sum()}): OK")

<!--SOL175184-->

**Ej. 3 — Holm** vs Bonferroni (Holm ≥ potencia).

In [ ]:
# <!--SOL175184-->
rej_h, adj_h, _, _ = multipletests(pvals, alpha=0.05, method="holm")
print(f"Bonferroni rechaza {rej_b.sum()}  Holm rechaza {rej_h.sum()}")
assert rej_h.sum() >= rej_b.sum()
print("Holm >= Bonferroni: OK")

<!--SOL175184-->

**Ej. 4 — BH / FDR.** 950 nulos `U(0,1)` + 50 alternativos concentrados cerca de 0 (p-values de un efecto real `z~N(3.5,1)`).

In [ ]:
# <!--SOL175184-->
m0, m1 = 950, 50
p_null = rng.uniform(0, 1, m0)
p_alt = 2 * stats.norm.sf(np.abs(rng.normal(3.5, 1.0, m1)))   # efecto real -> p pequenos
allp = np.concatenate([p_null, p_alt])
is_alt = np.concatenate([np.zeros(m0, bool), np.ones(m1, bool)])
rej, _, _, _ = multipletests(allp, alpha=0.05, method="fdr_bh")
disc = rej.sum(); fdr_emp = (rej & ~is_alt).sum()/max(disc, 1)
print(f"descubrimientos={disc}  FDR empirico={fdr_emp:.2%}")
assert disc > 0
print("BH controla el FDR: OK")

<!--SOL175184-->

**Ej. 5 — Comparación final:** rechazos, recall y FDR empírico.

In [ ]:
# <!--SOL175184-->
import pandas as pd
tabla = []
for met in ("bonferroni", "holm", "fdr_bh"):
    rj, _, _, _ = multipletests(allp, alpha=0.05, method=met)
    disc = rj.sum(); recall = (rj & is_alt).sum()/m1; fdr = (rj & ~is_alt).sum()/max(disc, 1)
    tabla.append([met, int(disc), f"{recall:.0%}", f"{fdr:.1%}"])
print(pd.DataFrame(tabla, columns=["metodo", "rechazos", "recall", "FDR_emp"]).to_string(index=False))
rec = {r[0]: r for r in tabla}
assert rec["fdr_bh"][1] >= rec["bonferroni"][1]
print("BH descubre mas con FDR controlado: OK")